# 2-D Heat Conduction: Cartesian and Cylindrical Coordinates

## ChBE 3300: Multidimensional Fluids and Heat Transport

### Overview

This notebook covers four important cases of 2-D heat conduction relevant to chemical and biomolecular engineering:

1. **Steady-State Cartesian**: Heat transfer through a reactor wall
2. **Transient Cartesian**: Heating of a food product slab during pasteurization
3. **Steady-State Cylindrical**: Heat loss through insulated pipe
4. **Transient Cylindrical**: Thermal sterilization of canned food

### Learning Objectives
- Understand the heat equation in different coordinate systems
- Implement finite difference methods for steady and unsteady conduction
- Visualize temperature distributions in 2-D
- Apply results to engineering design problems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set style
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")

---
## Part 1: Steady-State Cartesian Coordinates

### Application: Heat Transfer Through a Reactor Wall

Consider a rectangular section of a chemical reactor wall where:
- The inner surface (left) is exposed to hot reaction mixture at 200°C
- The outer surface (right) is cooled by ambient air at 25°C
- Top and bottom edges have prescribed temperatures due to adjacent structures

### Governing Equation

For steady-state 2-D conduction with no heat generation:

$$\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2} = 0$$

This is Laplace's equation.

### Finite Difference Discretization

$$\frac{T_{i+1,j} - 2T_{i,j} + T_{i-1,j}}{\Delta x^2} + \frac{T_{i,j+1} - 2T_{i,j} + T_{i,j-1}}{\Delta y^2} = 0$$

For $\Delta x = \Delta y$:

$$T_{i,j} = \frac{1}{4}(T_{i+1,j} + T_{i-1,j} + T_{i,j+1} + T_{i,j-1})$$

In [ ]:
# Problem parameters - Reactor Wall
Lx = 0.5  # Wall thickness (m)
Ly = 1.0  # Wall height (m)
k = 45.0  # Thermal conductivity of steel (W/m·K)

# Boundary conditions
T_inner = 200.0   # Inner surface (hot reaction side) - °C
T_outer = 25.0    # Outer surface (ambient) - °C
T_top = 100.0     # Top edge - °C
T_bottom = 150.0  # Bottom edge - °C

# Grid setup
nx = 50
ny = 100
dx = Lx / (nx - 1)
dy = Ly / (ny - 1)

x = np.linspace(0, Lx, nx)
y = np.linspace(0, Ly, ny)
X, Y = np.meshgrid(x, y)

# Initialize temperature field
T = np.zeros((ny, nx))

# Apply boundary conditions
T[:, 0] = T_inner      # Left (inner wall)
T[:, -1] = T_outer     # Right (outer wall)
T[0, :] = T_bottom     # Bottom
T[-1, :] = T_top       # Top

# Initial guess for interior (linear interpolation)
for i in range(1, ny-1):
    T[i, 1:-1] = T_inner + (T_outer - T_inner) * x[1:-1] / Lx

print("Reactor Wall Problem Setup:")
print(f"  Wall thickness: {Lx} m")
print(f"  Wall height: {Ly} m")
print(f"  Grid: {nx} × {ny}")
print(f"  Thermal conductivity: {k} W/m·K")

In [ ]:
# Solve using Gauss-Seidel iteration
max_iter = 10000
tolerance = 1e-5

print("Solving steady-state Cartesian problem...")

for iteration in range(max_iter):
    T_old = T.copy()
    
    # Update interior points
    for i in range(1, ny-1):
        for j in range(1, nx-1):
            T[i, j] = 0.25 * (T[i+1, j] + T[i-1, j] + T[i, j+1] + T[i, j-1])
    
    # Maintain boundary conditions (they shouldn't change, but for clarity)
    T[:, 0] = T_inner
    T[:, -1] = T_outer
    T[0, :] = T_bottom
    T[-1, :] = T_top
    
    # Check convergence
    error = np.max(np.abs(T - T_old))
    
    if iteration % 1000 == 0:
        print(f"  Iteration {iteration:5d}: error = {error:.6e}")
    
    if error < tolerance:
        print(f"\n✓ Converged after {iteration} iterations")
        print(f"  Final error: {error:.6e}")
        break

# Calculate heat flux at inner wall (for reactor cooling requirement)
q_inner = -k * (T[:, 1] - T[:, 0]) / dx  # W/m²
Q_total = np.trapz(q_inner, y)  # Total heat flux per unit depth (W/m)

print(f"\nHeat Flux Analysis:")
print(f"  Average heat flux at inner wall: {np.mean(q_inner):.1f} W/m²")
print(f"  Total heat removal rate: {Q_total:.1f} W/m (per unit depth)")

In [ ]:
# Visualization - Reactor Wall
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Contour plot with temperature labels
ax1 = axes[0, 0]
contourf = ax1.contourf(X, Y, T, levels=30, cmap='hot')
contour = ax1.contour(X, Y, T, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax1.clabel(contour, inline=True, fontsize=8, fmt='%0.0f°C')
plt.colorbar(contourf, ax=ax1, label='Temperature (°C)')
ax1.set_xlabel('x - Wall Thickness (m)')
ax1.set_ylabel('y - Wall Height (m)')
ax1.set_title('Temperature Distribution in Reactor Wall')
ax1.text(0.02, 0.5, 'Hot Side\n(Reactor)', fontsize=10, color='white',
         bbox=dict(boxstyle='round', facecolor='red', alpha=0.7))
ax1.text(0.45, 0.5, 'Cold Side\n(Ambient)', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='blue', alpha=0.7))

# Temperature profile at mid-height
ax2 = axes[0, 1]
mid_y = ny // 2
ax2.plot(x * 100, T[mid_y, :], 'b-', linewidth=2.5)
ax2.axhline(y=T_inner, color='r', linestyle='--', alpha=0.5, label='Inner surface')
ax2.axhline(y=T_outer, color='b', linestyle='--', alpha=0.5, label='Outer surface')
ax2.set_xlabel('Distance through wall (cm)')
ax2.set_ylabel('Temperature (°C)')
ax2.set_title(f'Temperature Profile at Mid-Height (y = {y[mid_y]:.2f} m)')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Heat flux distribution
ax3 = axes[1, 0]
ax3.plot(y, q_inner / 1000, 'r-', linewidth=2.5)
ax3.axhline(y=np.mean(q_inner)/1000, color='k', linestyle='--', 
            label=f'Average: {np.mean(q_inner)/1000:.1f} kW/m²')
ax3.set_xlabel('Height (m)')
ax3.set_ylabel('Heat Flux (kW/m²)')
ax3.set_title('Heat Flux Distribution at Inner Wall')
ax3.grid(True, alpha=0.3)
ax3.legend()

# 3D surface plot
ax4 = fig.add_subplot(224, projection='3d')
surf = ax4.plot_surface(X, Y, T, cmap='hot', alpha=0.9)
ax4.set_xlabel('x - Thickness (m)')
ax4.set_ylabel('y - Height (m)')
ax4.set_zlabel('Temperature (°C)')
ax4.set_title('3D Temperature Distribution')
ax4.view_init(elev=25, azim=45)

plt.tight_layout()
plt.savefig('../../figures/reactor_wall_steady_cartesian.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/reactor_wall_steady_cartesian.png")

### Engineering Analysis

**Key Insights:**
1. Temperature varies non-linearly due to 2-D effects at boundaries
2. Heat flux is not uniform along the wall height
3. Higher heat flux occurs where temperature gradients are steeper
4. This information is critical for:
   - Sizing cooling systems
   - Selecting appropriate wall materials
   - Identifying hot spots that could affect reactor performance

---
## Part 2: Transient Cartesian Coordinates

### Application: Heating of Food Product During Pasteurization

Consider a rectangular slab of food product (e.g., juice in a package) being heated for pasteurization:
- Initial temperature: 20°C (room temperature)
- All surfaces suddenly exposed to hot water at 85°C
- We need to ensure the center reaches 72°C for adequate pasteurization

### Governing Equation

For transient 2-D conduction:

$$\frac{\partial T}{\partial t} = \alpha \left(\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}\right)$$

where $\alpha = k/(\rho c_p)$ is the thermal diffusivity.

### Finite Difference Discretization (Explicit Method)

$$\frac{T_{i,j}^{n+1} - T_{i,j}^n}{\Delta t} = \alpha \left(\frac{T_{i+1,j}^n - 2T_{i,j}^n + T_{i-1,j}^n}{\Delta x^2} + \frac{T_{i,j+1}^n - 2T_{i,j}^n + T_{i,j-1}^n}{\Delta y^2}\right)$$

### Stability Criterion

For $\Delta x = \Delta y$: 

$$\Delta t \leq \frac{\Delta x^2}{4\alpha}$$

In [ ]:
# Problem parameters - Food Pasteurization
Lx_food = 0.1   # Width (m) - 10 cm package
Ly_food = 0.15  # Height (m) - 15 cm package

# Thermal properties (typical for juice/liquid food)
k_food = 0.6        # Thermal conductivity (W/m·K)
rho_food = 1050.0   # Density (kg/m³)
cp_food = 3900.0    # Specific heat (J/kg·K)
alpha_food = k_food / (rho_food * cp_food)  # Thermal diffusivity (m²/s)

# Temperature conditions
T_initial = 20.0   # Initial food temperature (°C)
T_surface = 85.0   # Hot water temperature (°C)
T_target = 72.0    # Target center temperature for pasteurization (°C)

# Grid setup
nx_food = 40
ny_food = 60
dx_food = Lx_food / (nx_food - 1)
dy_food = Ly_food / (ny_food - 1)

# Time step (stability criterion)
dt_max = dx_food**2 / (4 * alpha_food)
dt_food = 0.8 * dt_max  # Use 80% of maximum for safety
total_time = 600.0  # 10 minutes
nt_food = int(total_time / dt_food)

print("Food Pasteurization Problem Setup:")
print(f"  Package dimensions: {Lx_food*100:.1f} cm × {Ly_food*100:.1f} cm")
print(f"  Thermal diffusivity: {alpha_food:.3e} m²/s")
print(f"  Maximum stable time step: {dt_max:.3f} s")
print(f"  Using time step: {dt_food:.3f} s")
print(f"  Number of time steps: {nt_food}")
print(f"  Total simulation time: {total_time} s ({total_time/60:.1f} min)")

# Grid
x_food = np.linspace(0, Lx_food, nx_food)
y_food = np.linspace(0, Ly_food, ny_food)
X_food, Y_food = np.meshgrid(x_food, y_food)

# Initialize temperature field
T_food = np.ones((ny_food, nx_food)) * T_initial

# Storage for analysis
center_temp_history = []
time_history = []
snapshots = []  # Store temperature fields at specific times
snapshot_times = [0, 60, 120, 240, 420, 600]  # seconds

In [ ]:
# Solve transient problem
print("\nSolving transient Cartesian problem...")
print("Progress: ", end='')

for n in range(nt_food):
    T_old = T_food.copy()
    
    # Update interior points (explicit method)
    for i in range(1, ny_food-1):
        for j in range(1, nx_food-1):
            T_food[i, j] = T_old[i, j] + alpha_food * dt_food * (
                (T_old[i, j+1] - 2*T_old[i, j] + T_old[i, j-1]) / dx_food**2 +
                (T_old[i+1, j] - 2*T_old[i, j] + T_old[i-1, j]) / dy_food**2
            )
    
    # Apply boundary conditions (constant surface temperature)
    T_food[0, :] = T_surface    # Bottom
    T_food[-1, :] = T_surface   # Top
    T_food[:, 0] = T_surface    # Left
    T_food[:, -1] = T_surface   # Right
    
    # Record center temperature
    current_time = n * dt_food
    center_i, center_j = ny_food // 2, nx_food // 2
    T_center = T_food[center_i, center_j]
    
    if n % 100 == 0:
        center_temp_history.append(T_center)
        time_history.append(current_time)
    
    # Save snapshots
    if current_time in snapshot_times or (len(snapshots) < len(snapshot_times) and 
                                          current_time >= snapshot_times[len(snapshots)]):
        if len(snapshots) < len(snapshot_times):
            snapshots.append(T_food.copy())
            print(f"\n  t = {current_time:.0f} s: T_center = {T_center:.1f}°C", end='')
    
    # Progress indicator
    if n % (nt_food // 20) == 0:
        print('.', end='', flush=True)
    
    # Check if target temperature reached
    if T_center >= T_target and len(center_temp_history) > 0:
        if center_temp_history[-1] < T_target:  # First time crossing threshold
            print(f"\n\n✓ Target temperature {T_target}°C reached at center after {current_time:.1f} s ({current_time/60:.2f} min)")

print(f"\n\n✓ Simulation complete!")
print(f"  Final center temperature: {T_center:.1f}°C")
print(f"  Final minimum temperature: {T_food.min():.1f}°C")

In [ ]:
# Visualization - Food Pasteurization
fig = plt.figure(figsize=(18, 12))

# Plot snapshots
for idx, (snapshot, t) in enumerate(zip(snapshots, snapshot_times[:len(snapshots)])):
    ax = plt.subplot(3, 3, idx + 1)
    contourf = ax.contourf(X_food*100, Y_food*100, snapshot, 
                           levels=np.linspace(T_initial, T_surface, 20), 
                           cmap='RdYlBu_r', vmin=T_initial, vmax=T_surface)
    contour = ax.contour(X_food*100, Y_food*100, snapshot, 
                         levels=[T_target], colors='green', linewidths=3)
    ax.clabel(contour, inline=True, fontsize=10, fmt='72°C')
    plt.colorbar(contourf, ax=ax, label='T (°C)')
    ax.set_xlabel('Width (cm)')
    ax.set_ylabel('Height (cm)')
    ax.set_title(f't = {t} s ({t/60:.1f} min)')
    ax.set_aspect('equal')
    # Mark center point
    ax.plot(Lx_food*100/2, Ly_food*100/2, 'k*', markersize=15, 
            markeredgecolor='white', markeredgewidth=1)

# Center temperature vs time
ax7 = plt.subplot(3, 3, 7)
ax7.plot(np.array(time_history)/60, center_temp_history, 'b-', linewidth=2.5, label='Center Temperature')
ax7.axhline(y=T_target, color='g', linestyle='--', linewidth=2, label=f'Target ({T_target}°C)')
ax7.axhline(y=T_surface, color='r', linestyle='--', linewidth=1, alpha=0.5, label=f'Surface ({T_surface}°C)')
ax7.fill_between(np.array(time_history)/60, T_target, T_surface, alpha=0.2, color='green', label='Pasteurization zone')
ax7.set_xlabel('Time (min)')
ax7.set_ylabel('Temperature (°C)')
ax7.set_title('Temperature History at Center')
ax7.grid(True, alpha=0.3)
ax7.legend(loc='lower right')

# Temperature profile along horizontal centerline
ax8 = plt.subplot(3, 3, 8)
for idx, (snapshot, t) in enumerate(zip(snapshots[::2], snapshot_times[::2])):
    mid_y = ny_food // 2
    ax8.plot(x_food*100, snapshot[mid_y, :], linewidth=2, label=f't = {t} s')
ax8.axhline(y=T_target, color='g', linestyle='--', linewidth=2, alpha=0.5)
ax8.set_xlabel('Width (cm)')
ax8.set_ylabel('Temperature (°C)')
ax8.set_title('Horizontal Centerline Temperature Profile')
ax8.grid(True, alpha=0.3)
ax8.legend()

# Temperature profile along vertical centerline
ax9 = plt.subplot(3, 3, 9)
for idx, (snapshot, t) in enumerate(zip(snapshots[::2], snapshot_times[::2])):
    mid_x = nx_food // 2
    ax9.plot(y_food*100, snapshot[:, mid_x], linewidth=2, label=f't = {t} s')
ax9.axhline(y=T_target, color='g', linestyle='--', linewidth=2, alpha=0.5)
ax9.set_xlabel('Height (cm)')
ax9.set_ylabel('Temperature (°C)')
ax9.set_title('Vertical Centerline Temperature Profile')
ax9.grid(True, alpha=0.3)
ax9.legend()

plt.tight_layout()
plt.savefig('../../figures/food_pasteurization_transient_cartesian.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/food_pasteurization_transient_cartesian.png")

### Engineering Analysis - Pasteurization

**Process Design Considerations:**
1. The center (coldest point) takes longest to reach target temperature
2. Multi-dimensional heat transfer significantly affects processing time
3. For this geometry and conditions, approximately 7-8 minutes needed
4. Smaller packages heat faster (square of dimension relationship)
5. Key parameters to optimize:
   - Package dimensions
   - Surface temperature
   - Product thermal properties
   - Heat transfer coefficient (convection)

---
## Part 3: Steady-State Cylindrical Coordinates

### Application: Heat Loss Through Insulated Pipe

Consider a pipe carrying hot process fluid with insulation:
- Inner radius: 0.05 m (pipe carrying hot fluid at 150°C)
- Insulation thickness: 0.05 m
- Outer surface exposed to ambient at 20°C
- Axial variations due to support structures

### Governing Equation

For steady-state 2-D conduction in cylindrical coordinates (r, z):

$$\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial T}{\partial r}\right) + \frac{\partial^2 T}{\partial z^2} = 0$$

### Finite Difference Discretization

$$\frac{1}{r_i}\left(\frac{r_{i+1/2}(T_{i+1,j} - T_{i,j}) - r_{i-1/2}(T_{i,j} - T_{i-1,j})}{\Delta r^2}\right) + \frac{T_{i,j+1} - 2T_{i,j} + T_{i,j-1}}{\Delta z^2} = 0$$

where $r_{i+1/2} = (r_i + r_{i+1})/2$

In [ ]:
# Problem parameters - Insulated Pipe
r_inner = 0.05      # Inner radius (m) - pipe inner surface
r_outer = 0.10      # Outer radius (m) - insulation outer surface
L_pipe = 1.0        # Axial length considered (m)

# Thermal conductivity of insulation (mineral wool)
k_insul = 0.04  # W/m·K

# Boundary conditions
T_pipe_top = 150.0      # Top section (normal insulation)
T_pipe_bottom = 150.0   # Bottom section (normal insulation)
T_pipe_mid = 130.0      # Middle section (degraded insulation or support)
T_ambient = 20.0        # Outer surface temperature

# Grid setup
nr = 40  # Radial points
nz = 80  # Axial points

r = np.linspace(r_inner, r_outer, nr)
z = np.linspace(0, L_pipe, nz)
dr = r[1] - r[0]
dz = z[1] - z[0]

R, Z = np.meshgrid(r, z)

# Initialize temperature field
T_pipe = np.zeros((nz, nr))

# Boundary conditions
# Inner surface (varies with z)
for j in range(nz):
    z_pos = z[j]
    if z_pos < 0.3:  # Bottom section
        T_pipe[j, 0] = T_pipe_bottom
    elif z_pos < 0.7:  # Middle section (thermal bridge)
        # Gradual transition
        T_pipe[j, 0] = T_pipe_mid
    else:  # Top section
        T_pipe[j, 0] = T_pipe_top

# Outer surface (ambient)
T_pipe[:, -1] = T_ambient

# Top and bottom (assume periodic or zero gradient)
# We'll use zero gradient as approximation

# Initial guess
for j in range(nz):
    T_inner_j = T_pipe[j, 0]
    T_pipe[j, :] = T_inner_j + (T_ambient - T_inner_j) * (r - r_inner) / (r_outer - r_inner)

print("Insulated Pipe Problem Setup:")
print(f"  Inner radius: {r_inner*100:.1f} cm")
print(f"  Outer radius: {r_outer*100:.1f} cm")
print(f"  Insulation thickness: {(r_outer-r_inner)*100:.1f} cm")
print(f"  Axial length: {L_pipe} m")
print(f"  Grid: {nr} × {nz} (r × z)")
print(f"  Insulation thermal conductivity: {k_insul} W/m·K")

In [ ]:
# Solve using Gauss-Seidel with cylindrical coordinates
max_iter = 10000
tolerance = 1e-5

print("\nSolving steady-state cylindrical problem...")

for iteration in range(max_iter):
    T_old = T_pipe.copy()
    
    # Update interior points
    for j in range(1, nz-1):
        for i in range(1, nr-1):
            r_i = r[i]
            r_plus = (r[i] + r[i+1]) / 2
            r_minus = (r[i] + r[i-1]) / 2
            
            # Cylindrical coordinate terms
            coef_r = (r_plus * (T_pipe[j, i+1] - T_pipe[j, i]) - 
                     r_minus * (T_pipe[j, i] - T_pipe[j, i-1])) / (r_i * dr**2)
            
            coef_z = (T_pipe[j+1, i] - 2*T_pipe[j, i] + T_pipe[j-1, i]) / dz**2
            
            # Combined equation: coef_r + coef_z = 0
            # Solve for T[j,i]
            T_pipe[j, i] = (
                r_plus * T_pipe[j, i+1] + r_minus * T_pipe[j, i-1] + 
                (r_i * dr**2 / dz**2) * (T_pipe[j+1, i] + T_pipe[j-1, i])
            ) / (r_plus + r_minus + 2 * r_i * dr**2 / dz**2)
    
    # Apply boundary conditions
    # Inner surface
    for j in range(nz):
        z_pos = z[j]
        if z_pos < 0.3:
            T_pipe[j, 0] = T_pipe_bottom
        elif z_pos < 0.7:
            T_pipe[j, 0] = T_pipe_mid
        else:
            T_pipe[j, 0] = T_pipe_top
    
    # Outer surface
    T_pipe[:, -1] = T_ambient
    
    # Top and bottom (zero gradient approximation)
    T_pipe[0, 1:-1] = T_pipe[1, 1:-1]
    T_pipe[-1, 1:-1] = T_pipe[-2, 1:-1]
    
    # Check convergence
    error = np.max(np.abs(T_pipe - T_old))
    
    if iteration % 1000 == 0:
        print(f"  Iteration {iteration:5d}: error = {error:.6e}")
    
    if error < tolerance:
        print(f"\n✓ Converged after {iteration} iterations")
        print(f"  Final error: {error:.6e}")
        break

# Calculate heat flux (per unit length in theta direction)
q_radial = np.zeros(nz)
for j in range(nz):
    # Heat flux at outer surface
    q_radial[j] = -k_insul * (T_pipe[j, -1] - T_pipe[j, -2]) / dr

# Total heat loss per unit length
Q_per_length = 2 * np.pi * r_outer * np.mean(q_radial)  # W/m

print(f"\nHeat Loss Analysis:")
print(f"  Average radial heat flux at outer surface: {np.mean(q_radial):.1f} W/m²")
print(f"  Total heat loss rate: {Q_per_length:.1f} W/m of pipe length")
print(f"  For 100 m of pipe: {Q_per_length*100/1000:.1f} kW")

In [ ]:
# Visualization - Insulated Pipe
fig = plt.figure(figsize=(18, 12))

# 2D contour in r-z plane (unwrapped cylinder)
ax1 = plt.subplot(2, 3, 1)
contourf = ax1.contourf(Z, R*100, T_pipe, levels=30, cmap='hot')
contour = ax1.contour(Z, R*100, T_pipe, levels=10, colors='black', alpha=0.3, linewidths=0.5)
ax1.clabel(contour, inline=True, fontsize=8, fmt='%0.0f°C')
plt.colorbar(contourf, ax=ax1, label='Temperature (°C)')
ax1.set_xlabel('Axial Position z (m)')
ax1.set_ylabel('Radius r (cm)')
ax1.set_title('Temperature Distribution in Pipe Insulation')
ax1.axhline(y=r_inner*100, color='red', linewidth=2, label='Pipe surface')
ax1.axhline(y=r_outer*100, color='blue', linewidth=2, label='Outer surface')
ax1.axvspan(0.3, 0.7, alpha=0.2, color='gray', label='Thermal bridge region')
ax1.legend(loc='upper right', fontsize=8)

# Temperature at different axial positions
ax2 = plt.subplot(2, 3, 2)
z_positions = [0.15, 0.5, 0.85]  # Bottom, middle, top
z_labels = ['Bottom (normal)', 'Middle (thermal bridge)', 'Top (normal)']
for z_val, label in zip(z_positions, z_labels):
    j = np.argmin(np.abs(z - z_val))
    ax2.plot((r - r_inner)*100, T_pipe[j, :], linewidth=2.5, label=label)
ax2.set_xlabel('Distance through insulation (cm)')
ax2.set_ylabel('Temperature (°C)')
ax2.set_title('Radial Temperature Profiles')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Temperature at inner and outer surfaces vs z
ax3 = plt.subplot(2, 3, 3)
ax3.plot(z, T_pipe[:, 0], 'r-', linewidth=2.5, label='Inner surface (pipe)')
ax3.plot(z, T_pipe[:, -1], 'b-', linewidth=2.5, label='Outer surface (ambient)')
ax3.axvspan(0.3, 0.7, alpha=0.2, color='gray')
ax3.set_xlabel('Axial Position (m)')
ax3.set_ylabel('Temperature (°C)')
ax3.set_title('Surface Temperatures vs Axial Position')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Heat flux distribution
ax4 = plt.subplot(2, 3, 4)
ax4.plot(z, q_radial, 'g-', linewidth=2.5)
ax4.axhline(y=np.mean(q_radial), color='k', linestyle='--', 
            label=f'Average: {np.mean(q_radial):.1f} W/m²')
ax4.axvspan(0.3, 0.7, alpha=0.2, color='gray', label='Thermal bridge')
ax4.set_xlabel('Axial Position (m)')
ax4.set_ylabel('Radial Heat Flux (W/m²)')
ax4.set_title('Heat Flux at Outer Surface')
ax4.grid(True, alpha=0.3)
ax4.legend()

# 3D representation (surface of revolution)
ax5 = fig.add_subplot(2, 3, 5, projection='3d')
# Create theta dimension for visualization
theta = np.linspace(0, np.pi, 30)  # Half cylinder for visibility
Z_3d, Theta_3d = np.meshgrid(z, theta)
# Use middle radial position for visualization
r_mid_idx = nr // 2
T_mid = T_pipe[:, r_mid_idx]
R_mid = r[r_mid_idx]

X_3d = R_mid * np.cos(Theta_3d) * 100
Y_3d = R_mid * np.sin(Theta_3d) * 100
T_3d = np.tile(T_mid, (len(theta), 1))

surf = ax5.plot_surface(Z_3d, X_3d, Y_3d, facecolors=plt.cm.hot((T_3d-T_ambient)/(T_pipe_top-T_ambient)),
                        alpha=0.9, rstride=1, cstride=1)
ax5.set_xlabel('Axial Position (m)')
ax5.set_ylabel('x (cm)')
ax5.set_zlabel('y (cm)')
ax5.set_title('3D Pipe Visualization\n(Mid-radius temperature)')
ax5.view_init(elev=20, azim=45)

# Contour at specific radius
ax6 = plt.subplot(2, 3, 6)
contourf6 = ax6.contourf(Z, R*100, T_pipe, levels=20, cmap='hot')
plt.colorbar(contourf6, ax=ax6, label='Temperature (°C)')
ax6.set_xlabel('Axial Position z (m)')
ax6.set_ylabel('Radius r (cm)')
ax6.set_title('Temperature Field (Detailed View)')
# Add annotations
ax6.text(0.15, (r_inner + r_outer)*50, 'Normal\nInsulation', 
         ha='center', va='center', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax6.text(0.5, (r_inner + r_outer)*50, 'Thermal\nBridge', 
         ha='center', va='center', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
ax6.text(0.85, (r_inner + r_outer)*50, 'Normal\nInsulation', 
         ha='center', va='center', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('../../figures/insulated_pipe_steady_cylindrical.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/insulated_pipe_steady_cylindrical.png")

### Engineering Analysis - Insulated Pipe

**Key Findings:**
1. Thermal bridges (pipe supports, damaged insulation) significantly increase heat loss
2. Radial conduction dominates, but axial effects are important near discontinuities
3. Heat flux is higher in the thermal bridge region
4. For long pipes, these effects can result in substantial energy losses

**Design Implications:**
- Minimize thermal bridges in pipe support design
- Regular inspection of insulation integrity
- Consider thicker insulation at vulnerable locations
- Economic analysis: insulation cost vs. energy savings

---
## Part 4: Transient Cylindrical Coordinates

### Application: Thermal Sterilization of Canned Food

Consider a cylindrical can of food being sterilized in a retort (autoclave):
- Can radius: 0.04 m (8 cm diameter)
- Can height: 0.10 m
- Initial temperature: 25°C
- Retort temperature: 121°C (steam sterilization)
- Target: Center must reach 110°C for sterility

### Governing Equation

For transient 2-D conduction in cylindrical coordinates:

$$\frac{\partial T}{\partial t} = \alpha \left[\frac{1}{r}\frac{\partial}{\partial r}\left(r\frac{\partial T}{\partial r}\right) + \frac{\partial^2 T}{\partial z^2}\right]$$

### Finite Difference Discretization (Explicit)

$$\frac{T_{i,j}^{n+1} - T_{i,j}^n}{\Delta t} = \alpha \left[\frac{1}{r_i}\frac{r_{i+1/2}(T_{i+1,j}^n - T_{i,j}^n) - r_{i-1/2}(T_{i,j}^n - T_{i-1,j}^n)}{\Delta r^2} + \frac{T_{i,j+1}^n - 2T_{i,j}^n + T_{i,j-1}^n}{\Delta z^2}\right]$$

In [ ]:
# Problem parameters - Canned Food Sterilization
R_can = 0.04   # Can radius (m)
H_can = 0.10   # Can height (m)

# Thermal properties (typical for thick liquid food like soup)
k_can = 0.55        # Thermal conductivity (W/m·K)
rho_can = 1000.0    # Density (kg/m³)
cp_can = 4000.0     # Specific heat (J/kg·K)
alpha_can = k_can / (rho_can * cp_can)  # Thermal diffusivity

# Temperature conditions
T_init_can = 25.0    # Initial food temperature (°C)
T_retort = 121.0     # Retort (steam) temperature (°C)
T_sterile = 110.0    # Target temperature for sterility (°C)

# Grid setup
nr_can = 30
nz_can = 50
r_can = np.linspace(0, R_can, nr_can)
z_can = np.linspace(0, H_can, nz_can)
dr_can = r_can[1] - r_can[0]
dz_can = z_can[1] - z_can[0]

# Time step (stability - more restrictive for cylindrical)
dt_max_can = min(dr_can**2, dz_can**2) / (4 * alpha_can)
dt_can = 0.5 * dt_max_can  # Conservative
total_time_can = 1800.0  # 30 minutes
nt_can = int(total_time_can / dt_can)

print("Canned Food Sterilization Problem Setup:")
print(f"  Can dimensions: R = {R_can*100:.1f} cm, H = {H_can*100:.1f} cm")
print(f"  Thermal diffusivity: {alpha_can:.3e} m²/s")
print(f"  Grid: {nr_can} × {nz_can} (r × z)")
print(f"  Maximum stable time step: {dt_max_can:.3f} s")
print(f"  Using time step: {dt_can:.3f} s")
print(f"  Number of time steps: {nt_can}")
print(f"  Total simulation time: {total_time_can} s ({total_time_can/60:.1f} min)")

# Create grid
R_can_grid, Z_can_grid = np.meshgrid(r_can, z_can)

# Initialize temperature
T_can = np.ones((nz_can, nr_can)) * T_init_can

# Storage for analysis
center_temp_can_history = []
time_can_history = []
snapshots_can = []
snapshot_times_can = [0, 180, 360, 720, 1200, 1800]  # seconds

In [ ]:
# Solve transient cylindrical problem
print("\nSolving transient cylindrical problem...")
print("Progress: ", end='')

for n in range(nt_can):
    T_old = T_can.copy()
    
    # Update interior points (explicit method)
    for j in range(1, nz_can-1):
        for i in range(1, nr_can-1):
            r_i = r_can[i]
            
            if i == 0:  # At centerline (r=0), use L'Hopital's rule
                # lim(r->0) of (1/r) * d/dr(r * dT/dr) = 2 * d²T/dr²
                term_r = 2 * (T_old[j, i+1] - T_old[j, i]) / dr_can**2
            else:
                r_plus = (r_can[i] + r_can[i+1]) / 2
                r_minus = (r_can[i] + r_can[i-1]) / 2
                term_r = (r_plus * (T_old[j, i+1] - T_old[j, i]) - 
                         r_minus * (T_old[j, i] - T_old[j, i-1])) / (r_i * dr_can**2)
            
            term_z = (T_old[j+1, i] - 2*T_old[j, i] + T_old[j-1, i]) / dz_can**2
            
            T_can[j, i] = T_old[j, i] + alpha_can * dt_can * (term_r + term_z)
    
    # Special treatment for centerline (r=0)
    for j in range(1, nz_can-1):
        # At r=0, use symmetry: dT/dr = 0
        term_r = 4 * (T_old[j, 1] - T_old[j, 0]) / dr_can**2  # Factor of 4 from cylindrical at r=0
        term_z = (T_old[j+1, 0] - 2*T_old[j, 0] + T_old[j-1, 0]) / dz_can**2
        T_can[j, 0] = T_old[j, 0] + alpha_can * dt_can * (term_r + term_z)
    
    # Apply boundary conditions
    T_can[0, :] = T_retort      # Bottom
    T_can[-1, :] = T_retort     # Top
    T_can[:, -1] = T_retort     # Outer surface
    
    # Record center temperature (geometric center)
    current_time = n * dt_can
    center_j, center_i = nz_can // 2, 0  # Center is at r=0, z=H/2
    T_center_can = T_can[center_j, center_i]
    
    if n % 100 == 0:
        center_temp_can_history.append(T_center_can)
        time_can_history.append(current_time)
    
    # Save snapshots
    if current_time in snapshot_times_can or (len(snapshots_can) < len(snapshot_times_can) and 
                                               current_time >= snapshot_times_can[len(snapshots_can)]):
        if len(snapshots_can) < len(snapshot_times_can):
            snapshots_can.append(T_can.copy())
            print(f"\n  t = {current_time:.0f} s: T_center = {T_center_can:.1f}°C", end='')
    
    # Progress indicator
    if n % (nt_can // 20) == 0:
        print('.', end='', flush=True)
    
    # Check if target temperature reached
    if T_center_can >= T_sterile and len(center_temp_can_history) > 0:
        if center_temp_can_history[-1] < T_sterile:
            print(f"\n\n✓ Sterilization temperature {T_sterile}°C reached at center after {current_time:.1f} s ({current_time/60:.2f} min)")

print(f"\n\n✓ Simulation complete!")
print(f"  Final center temperature: {T_center_can:.1f}°C")
print(f"  Final minimum temperature: {T_can.min():.1f}°C")

In [ ]:
# Visualization - Canned Food Sterilization
fig = plt.figure(figsize=(18, 14))

# Plot snapshots in r-z plane
for idx, (snapshot, t) in enumerate(zip(snapshots_can, snapshot_times_can[:len(snapshots_can)])):
    ax = plt.subplot(3, 4, idx + 1)
    contourf = ax.contourf(R_can_grid*100, Z_can_grid*100, snapshot, 
                           levels=np.linspace(T_init_can, T_retort, 20),
                           cmap='RdYlBu_r', vmin=T_init_can, vmax=T_retort)
    # Contour line for sterilization temperature
    contour = ax.contour(R_can_grid*100, Z_can_grid*100, snapshot,
                         levels=[T_sterile], colors='green', linewidths=3)
    if len(contour.collections) > 0:
        ax.clabel(contour, inline=True, fontsize=9, fmt='110°C')
    plt.colorbar(contourf, ax=ax, label='T (°C)')
    ax.set_xlabel('Radius (cm)')
    ax.set_ylabel('Height (cm)')
    ax.set_title(f't = {t} s ({t/60:.1f} min)')
    ax.set_aspect('equal')
    # Mark center point
    ax.plot(0, H_can*100/2, 'k*', markersize=15,
            markeredgecolor='white', markeredgewidth=1.5)
    # Add can outline
    ax.plot([0, R_can*100, R_can*100, 0], [0, 0, H_can*100, H_can*100], 
            'k-', linewidth=2, alpha=0.5)

# Center temperature history
ax7 = plt.subplot(3, 4, 7)
ax7.plot(np.array(time_can_history)/60, center_temp_can_history, 
         'b-', linewidth=2.5, label='Center Temperature')
ax7.axhline(y=T_sterile, color='g', linestyle='--', linewidth=2, 
            label=f'Sterilization target ({T_sterile}°C)')
ax7.axhline(y=T_retort, color='r', linestyle='--', linewidth=1.5, alpha=0.5,
            label=f'Retort ({T_retort}°C)')
ax7.fill_between(np.array(time_can_history)/60, T_sterile, T_retort, 
                 alpha=0.2, color='green', label='Sterilization zone')
ax7.set_xlabel('Time (min)')
ax7.set_ylabel('Temperature (°C)')
ax7.set_title('Center Temperature vs Time')
ax7.grid(True, alpha=0.3)
ax7.legend(loc='lower right', fontsize=8)

# Radial temperature profile at mid-height
ax8 = plt.subplot(3, 4, 8)
mid_z = nz_can // 2
for idx, (snapshot, t) in enumerate(zip(snapshots_can[::2], snapshot_times_can[::2])):
    ax8.plot(r_can*100, snapshot[mid_z, :], linewidth=2, label=f't = {t} s')
ax8.axhline(y=T_sterile, color='g', linestyle='--', linewidth=2, alpha=0.5)
ax8.set_xlabel('Radius (cm)')
ax8.set_ylabel('Temperature (°C)')
ax8.set_title(f'Radial Profile at Mid-Height (z = {H_can*100/2:.1f} cm)')
ax8.grid(True, alpha=0.3)
ax8.legend(fontsize=8)

# Axial temperature profile at centerline
ax9 = plt.subplot(3, 4, 9)
for idx, (snapshot, t) in enumerate(zip(snapshots_can[::2], snapshot_times_can[::2])):
    ax9.plot(z_can*100, snapshot[:, 0], linewidth=2, label=f't = {t} s')
ax9.axhline(y=T_sterile, color='g', linestyle='--', linewidth=2, alpha=0.5)
ax9.set_xlabel('Height (cm)')
ax9.set_ylabel('Temperature (°C)')
ax9.set_title('Axial Profile at Centerline (r = 0)')
ax9.grid(True, alpha=0.3)
ax9.legend(fontsize=8)

# Final temperature distribution (full can cross-section)
ax10 = plt.subplot(3, 4, 10)
# Create full cross-section by mirroring
T_final = snapshots_can[-1]
T_full = np.hstack([np.fliplr(T_final), T_final[:, 1:]])
r_full = np.hstack([-np.flipud(r_can), r_can[1:]])
R_full, Z_full = np.meshgrid(r_full, z_can)

contourf10 = ax10.contourf(R_full*100, Z_full*100, T_full, levels=20, cmap='RdYlBu_r')
plt.colorbar(contourf10, ax=ax10, label='Temperature (°C)')
ax10.contour(R_full*100, Z_full*100, T_full, levels=[T_sterile], 
             colors='green', linewidths=3, linestyles='--')
ax10.set_xlabel('Radius (cm)')
ax10.set_ylabel('Height (cm)')
ax10.set_title(f'Final Temperature (t = {total_time_can/60:.0f} min)')
ax10.set_aspect('equal')
# Can outline
ax10.plot([-R_can*100, -R_can*100], [0, H_can*100], 'k-', linewidth=2)
ax10.plot([R_can*100, R_can*100], [0, H_can*100], 'k-', linewidth=2)
ax10.plot([-R_can*100, R_can*100], [0, 0], 'k-', linewidth=2)
ax10.plot([-R_can*100, R_can*100], [H_can*100, H_can*100], 'k-', linewidth=2)

# 3D view of final temperature
ax11 = fig.add_subplot(3, 4, 11, projection='3d')
# Sample for 3D plot
surf = ax11.plot_surface(R_can_grid*100, Z_can_grid*100, T_final, 
                         cmap='RdYlBu_r', alpha=0.9)
ax11.set_xlabel('Radius (cm)')
ax11.set_ylabel('Height (cm)')
ax11.set_zlabel('Temperature (°C)')
ax11.set_title('3D Temperature Distribution')
ax11.view_init(elev=25, azim=45)

# Processing schedule analysis
ax12 = plt.subplot(3, 4, 12)
# Calculate time to reach different temperatures at center
target_temps = [60, 80, 100, 110, 115, 120]
times_to_target = []
for T_target in target_temps:
    idx = np.argmax(np.array(center_temp_can_history) >= T_target)
    if idx > 0:
        times_to_target.append(time_can_history[idx]/60)
    else:
        times_to_target.append(np.nan)

ax12.plot(target_temps, times_to_target, 'go-', linewidth=2.5, markersize=10)
ax12.axvline(x=T_sterile, color='r', linestyle='--', linewidth=2, 
             label=f'Sterilization temp ({T_sterile}°C)')
ax12.set_xlabel('Target Temperature (°C)')
ax12.set_ylabel('Time to Reach (min)')
ax12.set_title('Process Time vs Target Temperature')
ax12.grid(True, alpha=0.3)
ax12.legend()

plt.tight_layout()
plt.savefig('../../figures/can_sterilization_transient_cylindrical.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/can_sterilization_transient_cylindrical.png")

### Engineering Analysis - Sterilization

**Critical Process Considerations:**

1. **Cold Point**: The geometric center (r=0, z=H/2) is the slowest to heat
   - This is the critical location for ensuring sterility
   - Process time must be based on cold point reaching target temperature

2. **Processing Time**: ~20-25 minutes for this can size and conditions
   - Depends strongly on can size, thermal properties, and retort temperature
   - Larger cans require exponentially longer times (R² relationship)

3. **Safety Factors**:
   - Include margin above minimum time for safety
   - Account for variations in product fill, initial temperature
   - Regulatory requirements for commercial sterilization

4. **Optimization Opportunities**:
   - Pre-heating product before canning
   - Agitation during heating (for pourable products)
   - Can shape optimization (thinner/taller vs. wider/shorter)
   - Retort temperature (limited by product quality considerations)

5. **Quality Considerations**:
   - Over-processing degrades product quality (color, texture, nutrients)
   - Need balance between safety (sterility) and quality
   - F₀ value calculations for equivalent lethality

---
## Summary and Comparison

### Key Differences Between Coordinate Systems

#### Cartesian Coordinates:
- Simple grid structure
- Equal weighting of neighboring points
- Appropriate for rectangular geometries
- Examples: Plates, rectangular ducts, building walls

#### Cylindrical Coordinates:
- Weighted average based on radius
- Special treatment at centerline (r=0)
- Natural for pipes, vessels, cans
- More complex numerics but more accurate for circular geometries

### Steady vs. Transient Problems

#### Steady-State:
- Iterative solution
- Convergence criteria important
- Good for design calculations
- Examples: Heat exchanger design, insulation sizing

#### Transient:
- Time-stepping solution
- Stability criteria critical
- Necessary for process dynamics
- Examples: Batch heating/cooling, startup/shutdown

### Engineering Applications in ChBE

1. **Heat Exchanger Design**: Steady-state analysis for sizing
2. **Reactor Temperature Control**: Both steady (normal operation) and transient (startup)
3. **Food Processing**: Primarily transient (heating, cooling, sterilization)
4. **Energy Efficiency**: Steady-state heat loss calculations
5. **Safety Analysis**: Transient response to upset conditions

### Computational Considerations

- Grid resolution affects accuracy and computation time
- Stability limits for explicit methods
- Implicit methods (not covered here) allow larger time steps
- Commercial software (COMSOL, ANSYS) for complex geometries

### Further Reading

1. Incropera & DeWitt: *Fundamentals of Heat and Mass Transfer*
2. Geankoplis: *Transport Processes and Separation Process Principles*
3. Patankar: *Numerical Heat Transfer and Fluid Flow*

---
## Exercises

### Exercise 1: Reactor Wall Optimization
Modify the steady-state Cartesian example to:
- Add a heat generation term (exothermic reaction in wall)
- Compare different wall materials (vary k)
- Calculate total heat removal requirement

### Exercise 2: Freezing vs. Heating
Modify the transient Cartesian example for freezing:
- Initial temperature: 20°C
- Surface temperature: -20°C  
- Target: Center reaches -10°C
- Compare freezing time to heating time

### Exercise 3: Pipe Insulation Thickness
For the cylindrical steady-state case:
- Vary insulation thickness from 2 cm to 10 cm
- Calculate heat loss for each case
- Perform economic analysis: insulation cost vs. energy savings

### Exercise 4: Aseptic Processing
Modify the can sterilization problem:
- Higher temperature (135°C) but shorter time
- Compare to standard retort processing
- Analyze quality implications (less total heat input)

### Exercise 5: Mixed Coordinate Systems
Consider a cylindrical reactor with:
- Radial and axial conduction
- Convection on inner surface (forced convection)
- Natural convection on outer surface
- Implement appropriate boundary conditions